# 05 — LLM-as-a-Judge avec MedGemma 4B

**Objectif :** utiliser MedGemma 4B comme juge pour évaluer la qualité des traductions médicales.

**Pipeline :**
1. MedGemma 4B attribue un score Likert 1–5 à chaque traduction
2. Évaluation de GPT-4o et RAG Mistral sur le corpus WMT24
3. Comparaison avec MEDCON, BLEU, et les annotations du médecin

## 0. Setup

In [ ]:
import subprocess, os
subprocess.run(['pip', 'install', '-q', '--upgrade', 'numpy'])
os.kill(os.getpid(), 9)

In [1]:
import os

if not os.path.exists('/content/Evaluating_medical_machine_translation'):
    !git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

Cloning into 'Evaluating_medical_machine_translation'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 92 (delta 41), reused 72 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 13.24 MiB | 5.10 MiB/s, done.
Resolving deltas: 100% (41/41), done.
Répertoire : /content/Evaluating_medical_machine_translation


In [2]:
from google.colab import files

os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Uploader : corpus_WMT24.json + unique_contexts_for_RAG.json + cleaned_mesh_snomed_dico.json + dr_annotations.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'   {dest}')


Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving corpus_WMT24.json to corpus_WMT24.json
Saving dr_annotations.json to dr_annotations.json
Saving unique_contexts_for_RAG.json to unique_contexts_for_RAG.json
   data/cleaned_mesh_snomed_dico.json
   data/corpus_WMT24.json
   data/dr_annotations.json
   data/unique_contexts_for_RAG.json


In [3]:
!pip uninstall mistralai -y
!pip install mistralai==1.2.5 --quiet
!pip install -q pyahocorasick sacrebleu sentence-transformers faiss-cpu transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00


## 1. Imports

In [18]:
import os
import sys
import json
import re
import time
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
from scipy.stats import pearsonr, spearmanr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
import huggingface_hub

# Login HuggingFace pour accéder à MedGemma
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
huggingface_hub.login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
print(f'Token : {userdata.get("HF_TOKEN")[:10]}...')  # vérifier qu'il est bien chargé

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')

print('Imports OK')
print(f'CUDA disponible : {torch.cuda.is_available()}')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Token : hf_pCmsqoa...
Imports OK
CUDA disponible : True


## 2. Chargement des données

In [5]:
with open('data/corpus_WMT24.json', encoding='utf-8') as f:
    test_docs = json.load(f)

with open('data/dr_annotations.json', encoding='utf-8') as f:
    annotations_raw = json.load(f)

# Extraction annotations médecin
medical_data = []
for doc in annotations_raw:
    if not doc['annotations']:
        continue
    ann = doc['annotations'][0]
    score = None
    for r in ann['result']:
        if r['from_name'] == 'translation_likert':
            score = int(r['value']['choices'][0])
    medical_data.append({
        'source_en'      : doc['data']['question'],
        'translation_fr' : doc['data']['translation'],
        'medical_score'  : score,
    })

print(f'Corpus WMT24        : {len(test_docs)} documents')
print(f'Annotations médecin : {len(medical_data)} documents')

Corpus WMT24        : 49 documents
Annotations médecin : 25 documents


## 2b. Régénération des traductions RAG Mistral

In [6]:
from mistralai import Mistral
from sentence_transformers import SentenceTransformer
import faiss
from google.colab import userdata

# Corpus RAG
with open('data/unique_contexts_for_RAG.json', encoding='utf-8') as f:
    rag_contexts = json.load(f)
rag_texts = [str(c) for c in rag_contexts]
print(f'Corpus RAG : {len(rag_texts)} documents')

# Index FAISS
print('Construction index FAISS...')
embed_model = SentenceTransformer('intfloat/multilingual-e5-base')
rag_embeddings = embed_model.encode(
    [f'passage: {t}' for t in rag_texts],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
index = faiss.IndexFlatIP(rag_embeddings.shape[1])
index.add(rag_embeddings.astype('float32'))
print(f'Index FAISS : {index.ntotal} vecteurs')

def retrieve_top_k(query, k=3):
    q = embed_model.encode([f'query: {query}'], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q, k)
    return [rag_texts[i] for i in indices[0]]

# Mistral
mistral_client = Mistral(api_key=userdata.get('MISTRAL_API_KEY'))

print('\nGénération traductions RAG...')
rag_translations = []
for doc in tqdm(test_docs, desc='RAG Mistral'):
    contexts = retrieve_top_k(doc['text_en'])
    context_block = '\n\n'.join([f'Example {i+1}:\n{c}' for i, c in enumerate(contexts)])
    prompt = f"""You are an expert medical translator. Translate the following English medical text into French.
To help you, here are examples of French medical texts from the same domain:

{context_block}

English text to translate:
{doc['text_en']}

Provide only the French translation, without any explanation or comment."""
    try:
        response = mistral_client.chat.complete(
            model='mistral-small-latest',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=1000
        )
        rag_translations.append(response.choices[0].message.content.strip())
    except Exception as e:
        print(f'Erreur : {e}')
        rag_translations.append('')
    time.sleep(0.3)

# Sauvegarde
with open('results/rag_translations.json', 'w', encoding='utf-8') as f:
    json.dump(rag_translations, f, ensure_ascii=False, indent=2)
print(f'\n{len(rag_translations)} traductions générées et sauvegardées ✓')


Corpus RAG : 10995 documents
Construction index FAISS...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/172 [00:00<?, ?it/s]

Index FAISS : 10995 vecteurs

Génération traductions RAG...


RAG Mistral:   0%|          | 0/49 [00:00<?, ?it/s]


49 traductions générées et sauvegardées ✓


## 3. Chargement MedGemma 4B

In [19]:
MODEL_NAME = 'google/medgemma-4b-it'

print(f'Chargement {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=userdata.get('HF_TOKEN'))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    token=userdata.get('HF_TOKEN')
)

# 4-bit quantization pour tenir sur Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()
print('MedGemma 4B chargé ')

Chargement google/medgemma-4b-it...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma 4B chargé ✓


## 4. Fonction de scoring LLM-as-a-judge

In [23]:
JUDGE_PROMPT = """You are an expert medical translator and evaluator.
Evaluate the quality of the following French translation of an English medical text.

English source:
{source_en}

French translation:
{translation_fr}

Rate the translation quality on a scale from 1 to 5:
- 5: Perfect translation, medically accurate, natural French, correct terminology
- 4: Good translation, minor style or terminology issues, clinically acceptable
- 3: Acceptable translation, some terminology errors or unnatural phrasing
- 2: Poor translation, significant terminology errors, potentially misleading
- 1: Very poor translation, major errors, clinically dangerous

Respond with a single integer between 1 and 5, nothing else."""


def judge_translation(source_en, translation_fr, max_new_tokens=10):
    prompt = JUDGE_PROMPT.format(
        source_en=source_en[:1000],
        translation_fr=translation_fr[:1000]
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_ids = inputs['input_ids']

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][input_ids.shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    match = re.search(r'[1-5]', text)
    if match:
        return int(match.group())
    else:
        print(f'Score non parsé : {repr(text)}')
        return None

# Test rapide
test_score = judge_translation(
    test_docs[0]['text_en'],
    test_docs[0]['gpt_translation']
)
print(f'Test score GPT-4o doc #0 : {test_score}/5')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Test score GPT-4o doc #0 : 2/5


## 5. Scoring MedGemma sur GPT-4o (corpus WMT24)

In [24]:
print('Scoring MedGemma — GPT-4o...')
scores_gpt = []
for doc in tqdm(test_docs, desc='MedGemma GPT-4o'):
    score = judge_translation(doc['text_en'], doc['gpt_translation'])
    scores_gpt.append(score)

valid_gpt = [s for s in scores_gpt if s is not None]
print(f'\nScores GPT-4o :')
print(f'  Moyenne : {np.mean(valid_gpt):.2f}/5')
print(f'  Std     : {np.std(valid_gpt):.2f}')
print(f'  Min/Max : {min(valid_gpt)}/{max(valid_gpt)}')
print(f'  Non parsés : {scores_gpt.count(None)}/{len(scores_gpt)}')

Scoring MedGemma — GPT-4o...


MedGemma GPT-4o:   0%|          | 0/49 [00:00<?, ?it/s]


Scores GPT-4o :
  Moyenne : 3.33/5
  Std     : 1.13
  Min/Max : 2/5
  Non parsés : 0/49


## 6. Scoring MedGemma sur RAG Mistral (corpus WMT24)

> Si tu n'as pas les traductions RAG en mémoire, relance le notebook 04 d'abord
> et copie `rag_translations` ici, ou uploade le fichier JSON des traductions.

In [25]:
# Les traductions RAG ont été générées et sauvegardées à l'étape 2b
# Si besoin de les recharger :
# with open('results/rag_translations.json', encoding='utf-8') as f:
#     rag_translations = json.load(f)

print(f'RAG translations disponibles : {len(rag_translations)} docs')


RAG translations disponibles : 49 docs


In [26]:
print('Scoring MedGemma — RAG Mistral...')
scores_rag = []
for i, doc in enumerate(tqdm(test_docs, desc='MedGemma RAG')):
    if rag_translations[i]:
        score = judge_translation(doc['text_en'], rag_translations[i])
    else:
        score = None
    scores_rag.append(score)

valid_rag = [s for s in scores_rag if s is not None]
print(f'\nScores RAG Mistral :')
print(f'  Moyenne : {np.mean(valid_rag):.2f}/5')
print(f'  Std     : {np.std(valid_rag):.2f}')
print(f'  Min/Max : {min(valid_rag)}/{max(valid_rag)}')
print(f'  Non parsés : {scores_rag.count(None)}/{len(scores_rag)}')

Scoring MedGemma — RAG Mistral...


MedGemma RAG:   0%|          | 0/49 [00:00<?, ?it/s]


Scores RAG Mistral :
  Moyenne : 3.45/5
  Std     : 1.11
  Min/Max : 2/5
  Non parsés : 0/49


## 7. Comparaison GPT-4o vs RAG — MedGemma judge

In [27]:
print('=' * 60)
print('COMPARAISON GPT-4o vs RAG — MedGemma as Judge')
print('=' * 60)
print(f"\n{'Système':<30} {'Score moyen':>12} {'Std':>8}")
print('-' * 52)
print(f"{'GPT-4o':<30} {np.mean(valid_gpt):>12.2f} {np.std(valid_gpt):>8.2f}")
print(f"{'RAG (Mistral + Embeddings)':<30} {np.mean(valid_rag):>12.2f} {np.std(valid_rag):>8.2f}")

# Distribution des scores
from collections import Counter
print('\n--- Distribution des scores ---')
print(f"{'Score':<8} {'GPT-4o':>10} {'RAG':>10}")
print('-' * 30)
for s in [1, 2, 3, 4, 5]:
    print(f"{s}/5    {scores_gpt.count(s):>10} {scores_rag.count(s):>10}")

# Delta doc par doc
deltas = [scores_rag[i] - scores_gpt[i]
          for i in range(len(test_docs))
          if scores_rag[i] is not None and scores_gpt[i] is not None]
print(f'\nDelta moyen (RAG - GPT-4o) : {np.mean(deltas):+.2f}')
print(f'Docs où RAG > GPT-4o : {sum(d > 0 for d in deltas)}/{len(deltas)}')
print(f'Docs où RAG < GPT-4o : {sum(d < 0 for d in deltas)}/{len(deltas)}')
print(f'Docs égaux           : {sum(d == 0 for d in deltas)}/{len(deltas)}')

COMPARAISON GPT-4o vs RAG — MedGemma as Judge

Système                         Score moyen      Std
----------------------------------------------------
GPT-4o                                 3.33     1.13
RAG (Mistral + Embeddings)             3.45     1.11

--- Distribution des scores ---
Score        GPT-4o        RAG
------------------------------
1/5             0          0
2/5            17         14
3/5             8          9
4/5            15         16
5/5             9         10

Delta moyen (RAG - GPT-4o) : +0.12
Docs où RAG > GPT-4o : 13/49
Docs où RAG < GPT-4o : 10/49
Docs égaux           : 26/49


## 8. Corrélation MedGemma vs MEDCON / BLEU sur GPT-4o

In [28]:
# Charger les résultats du notebook 01
import sacrebleu as sb
from medcon import build_pairs, build_automaton, medcon_grouped
import json

with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict = json.load(f)

pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

medcon_scores, bleu_scores = [], []
for doc in tqdm(test_docs, desc='MEDCON+BLEU'):
    r = medcon_grouped(doc['text_en'], doc['gpt_translation'], pairs, automaton_en, automaton_fr)
    medcon_scores.append(r['f1'])
    bleu_scores.append(sb.sentence_bleu(doc['gpt_translation'], [doc['translation_fr']]).score)

# Corrélations avec MedGemma
judge_scores = scores_gpt
valid_idx = [i for i in range(len(test_docs)) if judge_scores[i] is not None]

y_judge  = [judge_scores[i]  for i in valid_idx]
y_medcon = [medcon_scores[i] for i in valid_idx]
y_bleu   = [bleu_scores[i]   for i in valid_idx]

print('=' * 70)
print('CORRÉLATIONS MEDGEMMA vs MÉTRIQUES AUTOMATIQUES (GPT-4o)')
print('=' * 70)
print(f"\n{'Métrique':<18} {'Pearson r':>10} {'p-val':>8} {'Spearman rho':>14} {'p-val':>8}")
print('-' * 64)

for name, y in [('MEDCON F1', y_medcon), ('BLEU', y_bleu)]:
    r_p, p_p = pearsonr(y_judge, y)
    r_s, p_s = spearmanr(y_judge, y)
    print(f'{name:<18} {r_p:>10.3f} {p_p:>8.4f} {r_s:>14.3f} {p_s:>8.4f}')

MEDCON+BLEU:   0%|          | 0/49 [00:00<?, ?it/s]

CORRÉLATIONS MEDGEMMA vs MÉTRIQUES AUTOMATIQUES (GPT-4o)

Métrique            Pearson r    p-val   Spearman rho    p-val
----------------------------------------------------------------
MEDCON F1               0.043   0.7669          0.042   0.7736
BLEU                   -0.305   0.0329         -0.233   0.1073


## 9. Corrélation MedGemma vs annotations médecin

In [29]:
print('Scoring MedGemma — corpus annotations médecin...')
scores_medgemma_ann = []
for doc in tqdm(medical_data, desc='MedGemma annotations'):
    score = judge_translation(doc['source_en'], doc['translation_fr'])
    scores_medgemma_ann.append(score)

# Corrélation avec score médecin
valid_pairs = [
    (medical_data[i]['medical_score'], scores_medgemma_ann[i])
    for i in range(len(medical_data))
    if medical_data[i]['medical_score'] is not None and scores_medgemma_ann[i] is not None
]
y_doctor  = [p[0] for p in valid_pairs]
y_gemma   = [p[1] for p in valid_pairs]

r_p, p_p = pearsonr(y_doctor, y_gemma)
r_s, p_s = spearmanr(y_doctor, y_gemma)

print('=' * 70)
print('CORRÉLATION MEDGEMMA vs SCORE MÉDECIN')
print('=' * 70)
print(f'\n  Score médecin moyen   : {np.mean(y_doctor):.2f}/5')
print(f'  Score MedGemma moyen  : {np.mean(y_gemma):.2f}/5')
print(f'\n  Pearson r  : {r_p:+.3f}  (p={p_p:.4f})')
print(f'  Spearman ρ : {r_s:+.3f}  (p={p_s:.4f})')
print(f'\n  n = {len(valid_pairs)} documents')

# Distribution comparée
print('\n--- Distribution des scores ---')
print(f"{'Score':<8} {'Médecin':>10} {'MedGemma':>10}")
print('-' * 30)
from collections import Counter
doc_counts    = Counter(y_doctor)
gemma_counts  = Counter(y_gemma)
for s in [1, 2, 3, 4, 5]:
    print(f"{s}/5    {doc_counts.get(s, 0):>10} {gemma_counts.get(s, 0):>10}")

Scoring MedGemma — corpus annotations médecin...


MedGemma annotations:   0%|          | 0/25 [00:00<?, ?it/s]

CORRÉLATION MEDGEMMA vs SCORE MÉDECIN

  Score médecin moyen   : 4.32/5
  Score MedGemma moyen  : 3.96/5

  Pearson r  : +0.046  (p=0.8278)
  Spearman ρ : +0.008  (p=0.9699)

  n = 25 documents

--- Distribution des scores ---
Score       Médecin   MedGemma
------------------------------
1/5             0          0
2/5             0          1
3/5             0          2
4/5            17         19
5/5             8          3


## 10. Nettoyage metadata widgets (avant push)

In [30]:
import nbformat

for path in [
    'notebooks/01_medcon_evaluation.ipynb',
    'notebooks/02_dict_comparison.ipynb',
    'notebooks/03_doctor_annotations.ipynb',
    'notebooks/04_rag_embeddings.ipynb',
    'notebooks/05_llm_as_judge.ipynb',
]:
    if os.path.exists(path):
        nb = nbformat.read(path, as_version=4)
        if 'widgets' in nb.metadata:
            del nb.metadata['widgets']
        nbformat.write(nb, path)
        print(f'Nettoyé : {path}')

Nettoyé : notebooks/01_medcon_evaluation.ipynb
Nettoyé : notebooks/02_dict_comparison.ipynb
Nettoyé : notebooks/03_doctor_annotations.ipynb
Nettoyé : notebooks/04_rag_embeddings.ipynb
